# Bab 2 — Data dan Distribusi Sampling

**Buku:** *Practical Statistics for Data Scientists* (O'Reilly)

## Ringkasan Bab
Praktisi data science umumnya bekerja dengan sampel yang ditarik dari populasi yang lebih luas. Bab ini mengkaji tingkat keandalan statistik (seperti mean atau median) yang dihitung dari sampel:

1. **Distribusi Sampling** dan *Standard Error*.
2. **Teorema Limit Pusat (CLT)**.
3. **Metode Bootstrap** untuk resampling.
4. **Interval Kepercayaan**.
5. Berbagai **distribusi probabilitas** (normal, t-Student, binomial, Poisson, dll).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns

DATA = Path('..') / 'data'
sns.set_theme(style='whitegrid')
np.random.seed(seed=3)

## 1. Distribusi Sampling dari Suatu Statistik

**Teori.** Distribusi sampling menunjukkan variasi statistik dari berbagai pengambilan sampel berulang. Berdasarkan Teorema Limit Pusat (CLT), distribusi sampling dari mean akan mendekati distribusi normal seiring bertambahnya ukuran sampel, terlepas dari bentuk distribusi populasi aslinya.

In [ ]:
loans_income = pd.read_csv(DATA / 'loans_income.csv').squeeze('columns')

sample_data = pd.DataFrame({
    'income': loans_income.sample(1000),
    'type': 'Data',
})
sample_mean_05 = pd.DataFrame({
    'income': [loans_income.sample(5).mean() for _ in range(1000)],
    'type': 'Mean of 5',
})
sample_mean_20 = pd.DataFrame({
    'income': [loans_income.sample(20).mean() for _ in range(1000)],
    'type': 'Mean of 20',
})
results = pd.concat([sample_data, sample_mean_05, sample_mean_20])
results.groupby('type').income.agg(['mean', 'std'])

In [ ]:
g = sns.FacetGrid(results, col='type', col_wrap=1, height=2, aspect=2)
g.map(plt.hist, 'income', range=[0, 200000], bins=40)
g.set_axis_labels('Income', 'Count')
g.set_titles('{col_name}')
plt.show()

## 2. Metode Bootstrap

**Teori.** Bootstrap adalah teknik resampling untuk mengestimasi distribusi sampling langsung dari sampel data yang tersedia tanpa asumsi teoretis yang kaku, dilakukan dengan pengambilan sampel berulang *dengan pengembalian*.

In [ ]:
results = []
for _ in range(1000):
    sample = resample(loans_income)
    results.append(sample.median())
results = pd.Series(results)

print('Bootstrap statistics')
print(f'Original: {loans_income.median()}')
print(f'Bias: {results.mean() - loans_income.median()}')
print(f'Std. error: {results.std()}')

## 3. Interval Kepercayaan (Confidence Intervals)

**Teori.** Interval kepercayaan memberikan rentang nilai yang memuat parameter populasi dengan tingkat keyakinan tertentu berdasarkan distribusi sampling atau hasil bootstrap.

In [ ]:
np.random.seed(seed=3)
sample20 = resample(loans_income, n_samples=20, replace=False)

results = []
for _ in range(500):
    results.append(resample(sample20).mean())
results = pd.Series(results)

confidence_interval = list(results.quantile([0.05, 0.95]))
ax = results.plot.hist(bins=30, figsize=(4, 3), edgecolor='white')
ax.plot(confidence_interval, [55, 55], color='black')
for x in confidence_interval:
    ax.plot([x, x], [0, 65], color='black')
    ax.text(x, 70, f'{x:.0f}', horizontalalignment='center', verticalalignment='center')
ax.text(sum(confidence_interval) / 2, 60, '90% interval',
        horizontalalignment='center', verticalalignment='center')
mean_income = results.mean()
ax.plot([mean_income, mean_income], [0, 50], color='black', linestyle='--')
ax.text(mean_income, 10, f'Mean: {mean_income:.0f}',
        bbox=dict(facecolor='white', edgecolor='white', alpha=0.5),
        horizontalalignment='center', verticalalignment='center')
ax.set_ylim(0, 80)
ax.set_ylabel('Counts')
plt.tight_layout(); plt.show()

In [ ]:
m, se = sample20.mean(), stats.sem(sample20)
t_int = stats.t.interval(0.90, df=len(sample20) - 1, loc=m, scale=se)
print('Bootstrap 90% CI:', [round(x) for x in confidence_interval])
print('t-based 90% CI:  ', [round(x) for x in t_int])

## 4. Distribusi Normal dan Ekor Panjang

**Teori.** Distribusi normal berbentuk lonceng simetris. Namun, banyak data dunia nyata yang memiliki **ekor panjang** (*long-tailed*), di mana nilai ekstrem terjadi lebih sering daripada model normal.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
norm_sample = stats.norm.rvs(size=100)
stats.probplot(norm_sample, plot=ax)
ax.set_title('QQ-plot of a truly normal sample')
plt.tight_layout(); plt.show()

In [ ]:
sp500_px = pd.read_csv(DATA / 'sp500_data.csv.gz', index_col=0)
spy = sp500_px['SPY']
fig, ax = plt.subplots(figsize=(4, 4))
stats.probplot(np.log(spy[spy > 0]).diff().dropna(), plot=ax)
ax.set_title('QQ-plot of SPY daily log returns')
plt.tight_layout(); plt.show()

## 5. Distribusi Binomial, Poisson, dan Lainnya

In [ ]:
print('P(exactly 2 clicks in 5 trials, p=0.1):', stats.binom.pmf(2, n=5, p=0.1))
print('P(2 or fewer clicks):', stats.binom.cdf(2, n=5, p=0.1))

k = np.arange(0, 6)
plt.figure(figsize=(4, 3))
plt.bar(k, stats.binom.pmf(k, n=5, p=0.1))
plt.xlabel('Number of successes'); plt.ylabel('Probability')
plt.tight_layout(); plt.show()

In [ ]:
x = np.linspace(0.1, 15, 200)
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for df in (1, 3, 6):
    axes[0].plot(x, stats.chi2.pdf(x, df), label=f'df={df}')
axes[0].set_title('Chi-square'); axes[0].legend()
for d1, d2 in ((3, 10), (5, 20)):
    axes[1].plot(x, stats.f.pdf(x, d1, d2), label=f'df1={d1}, df2={d2}')
axes[1].set_title('F'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
poisson_sample = stats.poisson.rvs(2, size=100)
expon_sample = stats.expon.rvs(scale=1 / 0.2, size=100)
weibull_sample = stats.weibull_min.rvs(1.5, scale=5000, size=100)

print('Poisson(lambda=2)   mean/var:', poisson_sample.mean(), poisson_sample.var())
print('Exponential(rate=0.2) mean:  ', expon_sample.mean())
print('Weibull(shape=1.5)  mean:    ', weibull_sample.mean())

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
axes[0].hist(poisson_sample, bins=range(0, 9), edgecolor='white'); axes[0].set_title('Poisson')
axes[1].hist(expon_sample, bins=15, edgecolor='white'); axes[1].set_title('Exponential')
axes[2].hist(weibull_sample, bins=15, edgecolor='white'); axes[2].set_title('Weibull')
plt.tight_layout(); plt.show()